# MedGemma 1.5 4B LoRA for chest X-ray anatomy and pathology localization

This notebook fine-tunes google/medgemma-1.5-4b-it to read a frontal chest X-ray and return structured bounding-box annotations.

Two complementary instruction tasks are created from every image–JSON pair:

- anatomy_localization: thoracic anatomy region, laterality, and bounding box
- pathology_localization: image-level pathology labels, normal/abnormal status, and localized observations

Source XYXY coordinates are converted to integers on a resolution-independent 0–1000 scale. The origin is the upper-left corner. This avoids teaching the model a different coordinate range for every source image size.

The two supplied files are schema examples, not an adequate training dataset. Use a substantially larger dataset and keep the held-out subject groups untouched until final evaluation.

## 1. Imports and environment check

No package-installation commands are included. Restart the kernel first if the environment was changed.

In [ ]:
import csv
import gc
import json
import math
import os
import random
import re
from collections import Counter, defaultdict
from importlib.metadata import version
from pathlib import Path

import torch
from PIL import Image, ImageDraw, ImageFont
from torch.utils.data import Dataset

from huggingface_hub import notebook_login, whoami, is_offline_mode
from peft import LoraConfig, PeftModel, get_peft_model
from transformers import (
    AutoModelForImageTextToText,
    AutoProcessor,
    Trainer,
    TrainingArguments,
)
from transformers.trainer_utils import get_last_checkpoint

for package in ["torch", "transformers", "huggingface-hub", "peft", "accelerate", "pillow"]:
    try:
        print(f"{package}: {version(package)}")
    except Exception as exc:
        print(f"{package}: unavailable ({exc})")

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("BF16 supported:", torch.cuda.is_bf16_supported())
print("Hugging Face offline mode:", is_offline_mode())

## 2. Hugging Face authentication

MedGemma is gated. Accept its Hugging Face license before running this cell. The token is handled by the Hugging Face client and is not embedded in this notebook.

In [ ]:
notebook_login()
print("Logged in as:", whoami()["name"])

## 3. Configuration

The two directories are treated as source pools because their original membership is not patient-independent. The notebook combines them, removes repeated studies, and creates a deterministic patient-level train/test split. Each directory must contain paired files such as 50000230.jpg and 50000230.json. No validation split is created. OUTPUT_ROOT stores checkpoints, the final adapter, predictions, and metrics.

In [ ]:
MODEL_ID = "google/medgemma-1.5-4b-it"

TRAIN_DATA_ROOT = Path("/data/liangz2/openi/multi_kg/train")
TEST_DATA_ROOT = Path("/data/liangz2/openi/multi_kg/test")
OUTPUT_ROOT = Path("/data/liangz2/openi/midrc/medgemma15_4b_cxr_bbox_lora_rank32")

IMAGE_EXTENSIONS = [".jpg", ".jpeg", ".png"]
SEED = 42
# None preserves the original test-folder image proportion after combining the two pools.
# Set a value such as 0.20 to request an explicit patient-level test proportion.
TEST_FRACTION = None
MINIMUM_UNIQUE_SUBJECTS_FOR_TRAINING = 10

COORDINATE_SCALE = 1000
MIN_LOCALIZATION_QUALITY = 2
INCLUDE_FALLBACK_BOXES = False
MAX_LENGTH = 2048

NUM_TRAIN_EPOCHS = 5
PER_DEVICE_TRAIN_BATCH_SIZE = 1
PER_DEVICE_EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 16
LEARNING_RATE = 1e-4
WARMUP_RATIO = 0.05
DATALOADER_NUM_WORKERS = 2

LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05
LORA_MULTIMODAL_PROJECTOR = True
LORA_VISION_TOWER = False

RUN_GENERATION_EVALUATION = True
# A final adapter is written only after trainer.train() returns successfully.
# Keep this True to reuse it on later notebook runs instead of training again.
SKIP_TRAINING_IF_FINAL_ADAPTER_EXISTS = True
MAX_NEW_TOKENS = {
    "anatomy_localization": 1200,
    "pathology_localization": 600,
}
IOU_THRESHOLDS = [0.25, 0.50]

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
assert LORA_R == 32
if TEST_FRACTION is not None:
    assert 0 < TEST_FRACTION < 0.5
print("Source pool 1:", TRAIN_DATA_ROOT)
print("Source pool 2:", TEST_DATA_ROOT)
print("Output root:", OUTPUT_ROOT)

## 4. Discover and validate image–JSON pairs

The validator checks image dimensions, metadata dimensions, box ordering, image boundaries, localization quality, and patient identifiers. The two source pools are combined, duplicate studies are removed, and subjects are reassigned as intact groups. Exact duplicate boxes are removed later, while distinct boxes with the same semantic label are preserved.

In [ ]:
def find_image_for_json(json_path):
    annotation = json.loads(json_path.read_text(encoding="utf-8"))
    candidates = []
    image_name = annotation.get("image_name")
    if image_name:
        candidates.append(json_path.parent / image_name)
    for extension in IMAGE_EXTENSIONS:
        candidates.append(json_path.with_suffix(extension))
    for candidate in candidates:
        if candidate.is_file():
            return candidate, annotation
    raise FileNotFoundError(f"No image found for {json_path}; checked {[str(path) for path in candidates]}")


def discover_pairs(data_root):
    json_paths = sorted(Path(data_root).rglob("*.json"))
    if not json_paths:
        raise FileNotFoundError(f"No JSON files found under {data_root}")
    pairs = []
    for json_path in json_paths:
        image_path, annotation = find_image_for_json(json_path)
        pairs.append({
            "json_path": str(json_path),
            "image_path": str(image_path),
            "annotation": annotation,
        })
    return pairs


def box_is_usable(box_record):
    quality = int(box_record.get("localization_quality", 0) or 0)
    if quality < MIN_LOCALIZATION_QUALITY:
        return False
    if not INCLUDE_FALLBACK_BOXES and bool(box_record.get("is_fallback", False)):
        return False
    return True


def validate_xyxy(box, width, height, context):
    if not isinstance(box, (list, tuple)) or len(box) != 4:
        raise ValueError(f"{context}: bbox must contain four values, found {box}")
    x1, y1, x2, y2 = [float(value) for value in box]
    if not (0 <= x1 < x2 <= width and 0 <= y1 < y2 <= height):
        raise ValueError(
            f"{context}: invalid XYXY box {box} for image width={width}, height={height}"
        )


def validate_pair(pair):
    annotation = pair["annotation"]
    required = {"study_id", "subject_id", "image_name", "bbox_annotation"}
    missing = required - set(annotation)
    if missing:
        raise ValueError(f"{pair['json_path']}: missing required fields {sorted(missing)}")

    with Image.open(pair["image_path"]) as image_file:
        actual_width, actual_height = image_file.size
    metadata = annotation["bbox_annotation"].get("image_metadata", {})
    metadata_width = int(metadata.get("width", actual_width))
    metadata_height = int(metadata.get("height", actual_height))
    if (metadata_width, metadata_height) != (actual_width, actual_height):
        raise ValueError(
            f"{pair['json_path']}: metadata dimensions {(metadata_width, metadata_height)} "
            f"do not match image dimensions {(actual_width, actual_height)}"
        )

    for collection_name in ["anatomy_bboxes", "observation_bboxes"]:
        for index, box_record in enumerate(annotation["bbox_annotation"].get(collection_name, [])):
            if box_is_usable(box_record):
                validate_xyxy(
                    box_record.get("bbox"), actual_width, actual_height,
                    f"{pair['json_path']} {collection_name}[{index}]",
                )
    return {
        "width": actual_width,
        "height": actual_height,
        "anatomy_boxes": sum(
            box_is_usable(box) for box in annotation["bbox_annotation"].get("anatomy_bboxes", [])
        ),
        "pathology_boxes": sum(
            box_is_usable(box) for box in annotation["bbox_annotation"].get("observation_bboxes", [])
        ),
    }


def validate_all_pairs(pairs):
    study_ids = set()
    image_paths = set()
    summary = Counter()
    for pair in pairs:
        study_id = str(pair["annotation"]["study_id"])
        if study_id in study_ids:
            raise ValueError(f"Duplicate study_id: {study_id}")
        if pair["image_path"] in image_paths:
            raise ValueError(f"Duplicate image path: {pair['image_path']}")
        study_ids.add(study_id)
        image_paths.add(pair["image_path"])
        result = validate_pair(pair)
        summary["images"] += 1
        summary["anatomy_boxes"] += result["anatomy_boxes"]
        summary["pathology_boxes"] += result["pathology_boxes"]
        summary["normal_images"] += str(pair["annotation"].get("normal", "")).lower() == "yes"
    summary["subjects"] = len({str(pair["annotation"]["subject_id"]) for pair in pairs})
    return summary


def combine_and_deduplicate_studies(first_pool, second_pool):
    """Keep one copy of a repeated study; reject conflicting subject/image identities."""
    by_study = {}
    duplicate_count = 0
    for pair in list(first_pool) + list(second_pool):
        annotation = pair["annotation"]
        study_id = str(annotation["study_id"])
        if study_id not in by_study:
            by_study[study_id] = pair
            continue
        existing = by_study[study_id]
        existing_annotation = existing["annotation"]
        identity = (
            str(annotation.get("subject_id")),
            str(annotation.get("bbox_annotation", {}).get("image_id", annotation.get("image_name"))),
        )
        existing_identity = (
            str(existing_annotation.get("subject_id")),
            str(existing_annotation.get("bbox_annotation", {}).get("image_id", existing_annotation.get("image_name"))),
        )
        if identity != existing_identity:
            raise ValueError(
                f"Conflicting records share study_id {study_id}: {existing_identity} versus {identity}"
            )
        duplicate_count += 1
    return [by_study[study_id] for study_id in sorted(by_study)], duplicate_count


def subject_stratum(subject_pairs):
    """Coarse stratification keeps normal-only and pathology-containing patients represented."""
    has_pathology = any(
        str(pair["annotation"].get("normal", "no")).lower() != "yes"
        or bool(pair["annotation"].get("labels"))
        for pair in subject_pairs
    )
    return "pathology_present" if has_pathology else "normal_only"


def patient_grouped_train_test_split(all_pairs, test_fraction, seed):
    pairs_by_subject = defaultdict(list)
    for pair in all_pairs:
        pairs_by_subject[str(pair["annotation"]["subject_id"])].append(pair)
    if len(pairs_by_subject) < 2:
        raise RuntimeError("At least two unique subjects are required for a patient-level split.")

    subjects_by_stratum = defaultdict(list)
    for subject_id, subject_pairs in pairs_by_subject.items():
        subjects_by_stratum[subject_stratum(subject_pairs)].append(subject_id)

    rng = random.Random(seed)
    selected_test_subjects = set()
    for stratum, subject_ids in sorted(subjects_by_stratum.items()):
        subject_ids = sorted(subject_ids)
        rng.shuffle(subject_ids)
        if len(subject_ids) == 1:
            n_test = 0
        else:
            n_test = min(len(subject_ids) - 1, max(1, round(len(subject_ids) * test_fraction)))
        selected_test_subjects.update(subject_ids[:n_test])

    if not selected_test_subjects:
        candidates = sorted(pairs_by_subject, key=lambda subject: (len(pairs_by_subject[subject]), subject))
        selected_test_subjects.add(candidates[0])
    if len(selected_test_subjects) == len(pairs_by_subject):
        selected_test_subjects.remove(sorted(selected_test_subjects)[-1])

    train, test = [], []
    for subject_id in sorted(pairs_by_subject):
        destination = test if subject_id in selected_test_subjects else train
        destination.extend(pairs_by_subject[subject_id])
    return train, test


source_training_pairs = discover_pairs(TRAIN_DATA_ROOT)
source_test_pairs = discover_pairs(TEST_DATA_ROOT)
source_training_summary = validate_all_pairs(source_training_pairs)
source_test_summary = validate_all_pairs(source_test_pairs)

all_pairs, duplicate_studies_removed = combine_and_deduplicate_studies(
    source_training_pairs, source_test_pairs
)
inferred_test_fraction = len(source_test_pairs) / (len(source_training_pairs) + len(source_test_pairs))
effective_test_fraction = inferred_test_fraction if TEST_FRACTION is None else TEST_FRACTION
if not 0 < effective_test_fraction < 0.5:
    raise ValueError(
        f"The effective test fraction is {effective_test_fraction:.4f}; set TEST_FRACTION explicitly "
        "to a value between 0 and 0.5."
    )

training_pairs, test_pairs = patient_grouped_train_test_split(
    all_pairs, effective_test_fraction, SEED
)
training_validation_summary = validate_all_pairs(training_pairs)
test_validation_summary = validate_all_pairs(test_pairs)
training_subjects = {str(pair["annotation"]["subject_id"]) for pair in training_pairs}
test_subjects = {str(pair["annotation"]["subject_id"]) for pair in test_pairs}
training_studies = {str(pair["annotation"]["study_id"]) for pair in training_pairs}
test_studies = {str(pair["annotation"]["study_id"]) for pair in test_pairs}
assert training_subjects.isdisjoint(test_subjects)
assert training_studies.isdisjoint(test_studies)

print("Original source-pool summaries:")
print("  train folder:", dict(source_training_summary))
print("  test folder:", dict(source_test_summary))
print("Duplicate studies removed while combining:", duplicate_studies_removed)
print(f"Requested/effective test fraction: {effective_test_fraction:.4f}")
print("Patient-grouped training dataset:", dict(training_validation_summary))
print("Patient-grouped held-out test dataset:", dict(test_validation_summary))
print(f"Actual held-out image fraction: {len(test_pairs) / len(all_pairs):.4f}")
print("No subject or study overlap remains after reassignment.")

## 5. Convert annotations into deterministic training targets

Anatomy and pathology are trained as separate prompts. Normal cases produce an empty localized_findings list, which is essential negative supervision. Report text is not placed in the model prompt because it would leak the answer at inference time.

In [ ]:
def normalize_bbox(box, width, height, scale=COORDINATE_SCALE):
    x1, y1, x2, y2 = [float(value) for value in box]
    normalized = [
        round(scale * x1 / width),
        round(scale * y1 / height),
        round(scale * x2 / width),
        round(scale * y2 / height),
    ]
    normalized = [max(0, min(scale, int(value))) for value in normalized]
    if not (normalized[0] < normalized[2] and normalized[1] < normalized[3]):
        raise ValueError(f"Normalization collapsed box {box} to {normalized}")
    return normalized


def unique_sorted(items, key_function):
    unique = {}
    for item in items:
        unique[key_function(item)] = item
    return [unique[key] for key in sorted(unique)]


def build_anatomy_target(pair):
    annotation = pair["annotation"]
    metadata = annotation["bbox_annotation"]["image_metadata"]
    width, height = int(metadata["width"]), int(metadata["height"])
    structures = []
    for source in annotation["bbox_annotation"].get("anatomy_bboxes", []):
        if not box_is_usable(source):
            continue
        structures.append({
            "region": str(source.get("region", "")).strip(),
            "laterality": str(source.get("laterality", "unknown")).strip().lower(),
            "bbox": normalize_bbox(source["bbox"], width, height),
        })
    structures = unique_sorted(
        structures,
        lambda item: (item["region"].lower(), item["laterality"], tuple(item["bbox"])),
    )
    return {
        "coordinate_system": f"normalized_0_{COORDINATE_SCALE}_xyxy",
        "anatomy": structures,
    }


def build_pathology_target(pair):
    annotation = pair["annotation"]
    metadata = annotation["bbox_annotation"]["image_metadata"]
    width, height = int(metadata["width"]), int(metadata["height"])
    findings = []
    for source in annotation["bbox_annotation"].get("observation_bboxes", []):
        if not box_is_usable(source):
            continue
        findings.append({
            "finding": str(source.get("name", "")).strip(),
            "entities": sorted({str(value).strip() for value in source.get("obs_entities", []) if str(value).strip()}),
            "laterality": str(source.get("laterality", "unknown")).strip().lower(),
            "certainty": str(source.get("certainty", "unknown")).strip().lower(),
            "bbox": normalize_bbox(source["bbox"], width, height),
        })
    findings = unique_sorted(
        findings,
        lambda item: (
            item["finding"].lower(), tuple(item["entities"]), item["laterality"], tuple(item["bbox"])
        ),
    )
    image_labels = sorted({str(label).strip() for label in annotation.get("labels", []) if str(label).strip()})
    bbox_summary = annotation.get("pathology_bbox_summary", {})
    unlocalized_labels = sorted({
        str(label).strip() for label in bbox_summary.get("labels_without_bbox_names", []) if str(label).strip()
    })
    return {
        "coordinate_system": f"normalized_0_{COORDINATE_SCALE}_xyxy",
        "normal": "yes" if str(annotation.get("normal", "no")).lower() == "yes" else "no",
        "image_labels": image_labels,
        "localized_findings": findings,
        "unlocalized_labels": unlocalized_labels,
    }


SYSTEM_PROMPTS = {
    "anatomy_localization": (
        "You are a chest radiograph localization assistant. Identify the annotated thoracic anatomy. "
        f"Return only valid JSON with coordinate_system and anatomy. Each anatomy item must contain "
        f"region, laterality, and bbox. bbox is [x1,y1,x2,y2] in a 0-{COORDINATE_SCALE} coordinate "
        "system relative to the full image, with the origin at the upper-left."
    ),
    "pathology_localization": (
        "You are a chest radiograph localization assistant. Detect annotated radiographic findings "
        "and localize findings that have bounding boxes. Return only valid JSON with coordinate_system, "
        "normal, image_labels, localized_findings, and unlocalized_labels. Each localized finding must "
        f"contain finding, entities, laterality, certainty, and bbox. bbox is [x1,y1,x2,y2] in a "
        f"0-{COORDINATE_SCALE} coordinate system relative to the full image. Use empty arrays when no "
        "pathology is annotated."
    ),
}

USER_PROMPTS = {
    "anatomy_localization": "Localize the annotated thoracic anatomy in this frontal chest X-ray.",
    "pathology_localization": "Detect and localize the annotated pathology in this frontal chest X-ray.",
}


def compact_json(value):
    return json.dumps(value, ensure_ascii=False, separators=(",", ":"), sort_keys=False)


def records_from_pairs(selected_pairs):
    records = []
    for pair in selected_pairs:
        study_id = str(pair["annotation"]["study_id"])
        for task, target_builder in [
            ("anatomy_localization", build_anatomy_target),
            ("pathology_localization", build_pathology_target),
        ]:
            records.append({
                "id": f"{study_id}::{task}",
                "study_id": study_id,
                "subject_id": str(pair["annotation"]["subject_id"]),
                "image_path": pair["image_path"],
                "task": task,
                "answer": compact_json(target_builder(pair)),
            })
    return records


def describe_records(name, records):
    print(f"{name}: {len(records):,} task records from {len({r['study_id'] for r in records}):,} images")
    print("  tasks:", dict(sorted(Counter(record["task"] for record in records).items())))
    print("  subjects:", len({record["subject_id"] for record in records}))


# These targets can be inspected even when the folder contains only the two schema examples.
example_records = records_from_pairs(training_pairs[:2])
for record in example_records:
    print(record["id"])
    print(record["answer"][:1000] + ("..." if len(record["answer"]) > 1000 else ""))
    print()

## 6. Optional annotation visualization

This cell overlays the normalized ground-truth boxes after converting them back to pixels. It is a useful coordinate-system sanity check before training.

In [ ]:
COLORS = {
    "anatomy": "#00D4FF",
    "localized_findings": "#FF3B30",
}


def denormalize_bbox(box, width, height, scale=COORDINATE_SCALE):
    x1, y1, x2, y2 = box
    return [
        round(width * x1 / scale),
        round(height * y1 / scale),
        round(width * x2 / scale),
        round(height * y2 / scale),
    ]


def draw_target(pair, task, max_side=1200):
    target = build_anatomy_target(pair) if task == "anatomy_localization" else build_pathology_target(pair)
    with Image.open(pair["image_path"]) as image_file:
        image = image_file.convert("RGB")
    width, height = image.size
    draw = ImageDraw.Draw(image)
    collection_name = "anatomy" if task == "anatomy_localization" else "localized_findings"
    for item in target[collection_name]:
        pixel_box = denormalize_bbox(item["bbox"], width, height)
        label = item.get("region", item.get("finding", "finding"))
        draw.rectangle(pixel_box, outline=COLORS[collection_name], width=max(3, width // 700))
        draw.text((pixel_box[0] + 3, pixel_box[1] + 3), label, fill=COLORS[collection_name])
    image.thumbnail((max_side, max_side))
    return image


display(draw_target(training_pairs[0], "anatomy_localization"))
display(draw_target(training_pairs[0], "pathology_localization"))

## 7. Multimodal supervised dataset and response-only collator

Only assistant target tokens contribute to the loss. Prompt and padding tokens are masked. Do not add random crops, rotations, or horizontal flips unless the boxes and laterality labels are transformed identically.

In [ ]:
def conversation(record, include_assistant=True):
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPTS[record["task"]]}]},
        {"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": USER_PROMPTS[record["task"]]},
        ]},
    ]
    if include_assistant:
        messages.append({
            "role": "assistant",
            "content": [{"type": "text", "text": record["answer"]}],
        })
    return messages


class CxrBoundingBoxDataset(Dataset):
    def __init__(self, records, processor, max_length):
        self.records = records
        self.processor = processor
        self.max_length = max_length

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]
        prompt_text = self.processor.apply_chat_template(
            conversation(record, include_assistant=False),
            add_generation_prompt=True,
            tokenize=False,
        )
        full_text = self.processor.apply_chat_template(
            conversation(record, include_assistant=True),
            add_generation_prompt=False,
            tokenize=False,
        )
        with Image.open(record["image_path"]) as image_file:
            image = image_file.convert("RGB")
            full = self.processor(
                text=full_text,
                images=image,
                return_tensors="pt",
                truncation=True,
                max_length=self.max_length,
            )
            prompt = self.processor(
                text=prompt_text,
                images=image,
                return_tensors="pt",
                truncation=True,
                max_length=self.max_length,
            )
        item = {key: value.squeeze(0) for key, value in full.items()}
        labels = item["input_ids"].clone()
        prompt_length = min(prompt["input_ids"].shape[-1], labels.shape[-1])
        labels[:prompt_length] = -100
        if torch.all(labels == -100):
            raise ValueError(
                f"Target was fully truncated for {record['id']}; increase MAX_LENGTH above {self.max_length}."
            )
        item["labels"] = labels
        return item


class MultimodalResponseOnlyCollator:
    def __init__(self, processor):
        tokenizer = processor.tokenizer
        if tokenizer.pad_token_id is None:
            tokenizer.pad_token = tokenizer.eos_token
        self.pad_token_id = tokenizer.pad_token_id

    @staticmethod
    def pad_first_dimension(tensor, target_length, value):
        if tensor.shape[0] == target_length:
            return tensor
        shape = (target_length - tensor.shape[0],) + tuple(tensor.shape[1:])
        padding = torch.full(shape, value, dtype=tensor.dtype)
        return torch.cat([tensor, padding], dim=0)

    def __call__(self, features):
        batch = {}
        sequence_keys = {"input_ids", "attention_mask", "token_type_ids", "labels"}
        max_length = max(feature["input_ids"].shape[0] for feature in features)
        for key in sequence_keys:
            if all(key in feature for feature in features):
                value = -100 if key == "labels" else (self.pad_token_id if key == "input_ids" else 0)
                batch[key] = torch.stack([
                    self.pad_first_dimension(feature[key], max_length, value) for feature in features
                ])
        for key in features[0]:
            if key in sequence_keys:
                continue
            values = [feature[key] for feature in features]
            batch[key] = torch.stack(values)
        return batch

## 8. Load MedGemma and attach rank-32 LoRA

The vision tower remains frozen by default. LoRA is attached to the language decoder and, when present, the multimodal projector. Set LORA_VISION_TOWER to True only when the dataset is large enough to justify additional spatial adaptation.

In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_ID)
if processor.tokenizer.pad_token_id is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token


def select_lora_targets(model):
    language_leaves = {"q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"}
    language_markers = ("language_model", "text_model")
    projector_markers = ("multi_modal_projector", "multimodal_projector", "mm_projector", "projector")
    vision_markers = ("vision_tower", "vision_model")
    targets = []
    categories = Counter()
    for name, module in model.named_modules():
        if not isinstance(module, torch.nn.Linear):
            continue
        leaf = name.rsplit(".", 1)[-1]
        lower_name = name.lower()
        if leaf in language_leaves and any(marker in lower_name for marker in language_markers):
            targets.append(name)
            categories["language"] += 1
        elif LORA_MULTIMODAL_PROJECTOR and any(marker in lower_name for marker in projector_markers):
            targets.append(name)
            categories["projector"] += 1
        elif LORA_VISION_TOWER and leaf in language_leaves and any(marker in lower_name for marker in vision_markers):
            targets.append(name)
            categories["vision"] += 1
    targets = sorted(set(targets))
    if not targets:
        raise RuntimeError("No LoRA targets were found. Inspect model.named_modules() for this Transformers version.")
    print("LoRA target categories:", dict(categories))
    print("Target examples:", targets[:12])
    return targets


def build_base_model():
    if not torch.cuda.is_available():
        raise RuntimeError("A CUDA GPU is required for this BF16 4B training notebook.")
    if not torch.cuda.is_bf16_supported():
        raise RuntimeError("The selected GPU does not report BF16 support.")
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
    model.config.use_cache = False
    if hasattr(model, "gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={"use_reentrant": False}
        )
    if hasattr(model, "enable_input_require_grads"):
        model.enable_input_require_grads()
    return model


def build_lora_model(adapter_path=None, is_trainable=True):
    model = build_base_model()
    if adapter_path is not None:
        print(f"Loading completed LoRA adapter from: {adapter_path}")
        model = PeftModel.from_pretrained(model, str(adapter_path), is_trainable=is_trainable)
        model.print_trainable_parameters()
        return model

    config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=select_lora_targets(model),
    )
    model = get_peft_model(model, config)
    model.print_trainable_parameters()
    return model


def model_input_device(model):
    for parameter in model.parameters():
        if parameter.device.type not in {"meta", "cpu"}:
            return parameter.device
    return next(model.parameters()).device

## 9. Train with the patient-grouped partition

The in-memory training and test sets come from combining both source directories and reassigning complete patient groups. Because no validation set is created, epoch-by-epoch evaluation, early stopping, and best-checkpoint selection are disabled. The patient-independent held-out test set is evaluated only after training is complete.

Trainer checkpoints and the final LoRA adapter serve different purposes. An interrupted training run resumes from `checkpoints/checkpoint-*`, which contains optimizer, scheduler, RNG, and Trainer state. Once training finishes and `final_adapter/adapter_config.json` exists, later notebook runs skip training and load that adapter directly for evaluation. To train a new model, use a new `OUTPUT_ROOT` (recommended) or explicitly disable `SKIP_TRAINING_IF_FINAL_ADAPTER_EXISTS`.

In [ ]:
if len(training_subjects) < MINIMUM_UNIQUE_SUBJECTS_FOR_TRAINING:
    raise RuntimeError(
        f"Only {len(training_subjects)} unique training subjects were found. The supplied examples are "
        f"suitable for schema validation but not fine-tuning. Provide at least "
        f"{MINIMUM_UNIQUE_SUBJECTS_FOR_TRAINING} subjects, and preferably hundreds or more."
    )

train_records = records_from_pairs(training_pairs)
test_records = records_from_pairs(test_pairs)

describe_records("Train", train_records)
describe_records("Held-out test", test_records)

split_subjects = {
    "train": {record["subject_id"] for record in train_records},
    "test": {record["subject_id"] for record in test_records},
}
assert split_subjects["train"].isdisjoint(split_subjects["test"])

manifest = {
    split: [
        {"id": record["id"], "study_id": record["study_id"], "subject_id": record["subject_id"], "task": record["task"]}
        for record in records
    ]
    for split, records in {
        "train": train_records,
        "test": test_records,
    }.items()
}
with (OUTPUT_ROOT / "dataset_split_manifest.json").open("w", encoding="utf-8") as handle:
    json.dump(manifest, handle, indent=2)

checkpoint_dir = OUTPUT_ROOT / "checkpoints"
preferred_adapter_dir = OUTPUT_ROOT / "final_adapter"
# Support both the notebook's final_adapter subdirectory and a legacy adapter saved directly in OUTPUT_ROOT.
existing_adapter_dir = next(
    (path for path in [preferred_adapter_dir, OUTPUT_ROOT] if (path / "adapter_config.json").is_file()),
    None,
)
training_already_complete = SKIP_TRAINING_IF_FINAL_ADAPTER_EXISTS and existing_adapter_dir is not None
if training_already_complete:
    print(f"Completed adapter found at {existing_adapter_dir}; training will be skipped.")
    model = build_lora_model(existing_adapter_dir, is_trainable=False)
    adapter_dir = existing_adapter_dir
else:
    model = build_lora_model()
    adapter_dir = preferred_adapter_dir
train_dataset = CxrBoundingBoxDataset(train_records, processor, MAX_LENGTH)
test_dataset = CxrBoundingBoxDataset(test_records, processor, MAX_LENGTH)
collator = MultimodalResponseOnlyCollator(processor)

# Fail early on schema, truncation, tensor shape, and forward-pass problems.
probe = collator([train_dataset[0]])
device = model_input_device(model)
probe = {
    key: value.to(device=device, dtype=torch.bfloat16) if value.is_floating_point() else value.to(device)
    for key, value in probe.items()
}
with torch.no_grad():
    print("Forward-pass probe loss:", model(**probe).loss.item())
del probe
torch.cuda.empty_cache()

training_args = TrainingArguments(
    output_dir=str(checkpoint_dir),
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    logging_steps=10,
    eval_strategy="no",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=False,
    bf16=True,
    fp16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="adamw_torch",
    dataloader_num_workers=DATALOADER_NUM_WORKERS,
    dataloader_pin_memory=True,
    remove_unused_columns=False,
    prediction_loss_only=True,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collator,
)
last_checkpoint = None
trained_this_run = False
if training_already_complete:
    print("Skipping trainer.train(); using the completed adapter for held-out evaluation.")
else:
    last_checkpoint = get_last_checkpoint(str(checkpoint_dir)) if checkpoint_dir.exists() else None
    if last_checkpoint:
        print("Resuming interrupted training from:", last_checkpoint)
    else:
        print("No Trainer checkpoint found; starting a new training run.")
    trainer.train(resume_from_checkpoint=last_checkpoint)
    trained_this_run = True
    trainer.save_model(str(adapter_dir))
    processor.save_pretrained(str(adapter_dir))

test_teacher_forced_loss = trainer.evaluate(test_dataset, metric_key_prefix="test_teacher_forced")

if trained_this_run:
    history_keys = sorted({key for row in trainer.state.log_history for key in row})
    with (OUTPUT_ROOT / "trainer_history.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=history_keys)
        writer.writeheader()
        writer.writerows(trainer.state.log_history)

training_status = {
    "model_id": MODEL_ID,
    "adapter_dir": str(adapter_dir),
    "training_skipped_because_final_adapter_exists": training_already_complete,
    "resumed_from_checkpoint": last_checkpoint,
    "trainer_global_step": trainer.state.global_step,
    "trainer_epoch": trainer.state.epoch,
    "test_teacher_forced_metrics": test_teacher_forced_loss,
}
with (OUTPUT_ROOT / "training_status.json").open("w", encoding="utf-8") as handle:
    json.dump(training_status, handle, indent=2)

print(test_teacher_forced_loss)

## 10. Generated-box evaluation

Evaluation requires valid JSON and uses label-aware, one-to-one greedy matching. A prediction matches a reference only when its semantic label and laterality match and its intersection-over-union reaches the selected threshold. Metrics include precision, recall, F1, mean matched IoU, normality accuracy, and image-label set scores.

Generation is restart-safe. Each completed prediction is appended immediately to `test_predictions.jsonl`. Predictions are organized into configurable evaluation batches. At the end of every batch, the notebook synchronizes the prediction checkpoint, recomputes cumulative metrics from all completed records, updates both `evaluation_metrics_partial.json` and `evaluation_metrics.json`, and appends a durable snapshot to `evaluation_metrics_history.jsonl`. If the cell or job is interrupted, rerun the notebook and this cell will validate the checkpoint, skip completed records, and continue with the first unfinished record. Set `RESET_GENERATION_CHECKPOINT = True` only when intentionally evaluating a newly trained adapter in the same output directory.

In [ ]:
def extract_json_object(text):

    cleaned = text.strip()
    start = cleaned.find("{")
    if start < 0:
        raise ValueError("No JSON object found")
    obj, _ = json.JSONDecoder().raw_decode(cleaned[start:])
    return obj


@torch.inference_mode()
def generate_record(model, record):
    prompt_text = processor.apply_chat_template(
        conversation(record, include_assistant=False),
        add_generation_prompt=True,
        tokenize=False,
    )
    with Image.open(record["image_path"]) as image_file:
        image = image_file.convert("RGB")
        inputs = processor(text=prompt_text, images=image, return_tensors="pt")
    device = model_input_device(model)
    moved = {
        key: value.to(device=device, dtype=torch.bfloat16) if value.is_floating_point() else value.to(device)
        for key, value in inputs.items()
    }
    input_length = moved["input_ids"].shape[-1]
    output_ids = model.generate(
        **moved,
        do_sample=False,
        max_new_tokens=MAX_NEW_TOKENS[record["task"]],
        pad_token_id=processor.tokenizer.pad_token_id,
        eos_token_id=processor.tokenizer.eos_token_id,
    )
    return processor.decode(output_ids[0, input_length:], skip_special_tokens=True).strip()


def valid_box(box):
    try:
        x1, y1, x2, y2 = [float(value) for value in box]
        return 0 <= x1 < x2 <= COORDINATE_SCALE and 0 <= y1 < y2 <= COORDINATE_SCALE
    except (TypeError, ValueError):
        return False


def bbox_iou(first, second):
    ax1, ay1, ax2, ay2 = [float(value) for value in first]
    bx1, by1, bx2, by2 = [float(value) for value in second]
    intersection_width = max(0.0, min(ax2, bx2) - max(ax1, bx1))
    intersection_height = max(0.0, min(ay2, by2) - max(ay1, by1))
    intersection = intersection_width * intersection_height
    area_a = (ax2 - ax1) * (ay2 - ay1)
    area_b = (bx2 - bx1) * (by2 - by1)
    union = area_a + area_b - intersection
    return intersection / union if union > 0 else 0.0


def normalized_text(value):
    return re.sub(r"\s+", " ", str(value).strip().lower())


def detection_key(item, task):
    laterality = normalized_text(item.get("laterality", "unknown"))
    if task == "anatomy_localization":
        return normalized_text(item.get("region", "")), laterality
    entities = tuple(sorted(normalized_text(value) for value in item.get("entities", []) if normalized_text(value)))
    finding = normalized_text(item.get("finding", ""))
    return (entities if entities else (finding,)), laterality


def collection_for_task(obj, task):
    key = "anatomy" if task == "anatomy_localization" else "localized_findings"
    collection = obj.get(key, []) if isinstance(obj, dict) else []
    return [item for item in collection if isinstance(item, dict) and valid_box(item.get("bbox"))]


def match_detections(reference_items, predicted_items, task, iou_threshold):
    candidates = []
    for reference_index, reference in enumerate(reference_items):
        for predicted_index, prediction in enumerate(predicted_items):
            if detection_key(reference, task) == detection_key(prediction, task):
                iou = bbox_iou(reference["bbox"], prediction["bbox"])
                if iou >= iou_threshold:
                    candidates.append((iou, reference_index, predicted_index))
    candidates.sort(reverse=True)
    used_reference, used_prediction, matched_ious = set(), set(), []
    for iou, reference_index, predicted_index in candidates:
        if reference_index in used_reference or predicted_index in used_prediction:
            continue
        used_reference.add(reference_index)
        used_prediction.add(predicted_index)
        matched_ious.append(iou)
    tp = len(matched_ious)
    return {
        "tp": tp,
        "fp": len(predicted_items) - tp,
        "fn": len(reference_items) - tp,
        "matched_ious": matched_ious,
    }


def safe_divide(numerator, denominator):
    return numerator / denominator if denominator else float("nan")


def set_counts(reference_values, predicted_values):
    reference = {normalized_text(value) for value in reference_values}
    predicted = {normalized_text(value) for value in predicted_values}
    return len(reference & predicted), len(predicted - reference), len(reference - predicted)


def evaluate_predictions(rows):
    results = {"valid_json_rate": safe_divide(sum(row["parsed_prediction"] is not None for row in rows), len(rows))}
    for task in ["anatomy_localization", "pathology_localization"]:
        task_rows = [row for row in rows if row["task"] == task]
        task_result = {"n_records": len(task_rows)}
        for threshold in IOU_THRESHOLDS:
            totals = Counter()
            matched_ious = []
            for row in task_rows:
                reference = json.loads(row["ground_truth"])
                prediction = row["parsed_prediction"] or {}
                matched = match_detections(
                    collection_for_task(reference, task),
                    collection_for_task(prediction, task),
                    task,
                    threshold,
                )
                totals.update({key: matched[key] for key in ["tp", "fp", "fn"]})
                matched_ious.extend(matched["matched_ious"])
            precision = safe_divide(totals["tp"], totals["tp"] + totals["fp"])
            recall = safe_divide(totals["tp"], totals["tp"] + totals["fn"])
            task_result[f"iou_{threshold:.2f}"] = {
                "tp": totals["tp"], "fp": totals["fp"], "fn": totals["fn"],
                "precision": precision,
                "recall": recall,
                "f1": safe_divide(2 * precision * recall, precision + recall),
                "mean_matched_iou": safe_divide(sum(matched_ious), len(matched_ious)),
            }
        if task == "pathology_localization":
            normal_correct = 0
            label_totals = Counter()
            for row in task_rows:
                reference = json.loads(row["ground_truth"])
                prediction = row["parsed_prediction"] or {}
                normal_correct += normalized_text(reference.get("normal")) == normalized_text(prediction.get("normal"))
                tp, fp, fn = set_counts(reference.get("image_labels", []), prediction.get("image_labels", []))
                label_totals.update({"tp": tp, "fp": fp, "fn": fn})
            label_precision = safe_divide(label_totals["tp"], label_totals["tp"] + label_totals["fp"])
            label_recall = safe_divide(label_totals["tp"], label_totals["tp"] + label_totals["fn"])
            task_result["normal_accuracy"] = safe_divide(normal_correct, len(task_rows))
            task_result["image_label_precision"] = label_precision
            task_result["image_label_recall"] = label_recall
            task_result["image_label_f1"] = safe_divide(
                2 * label_precision * label_recall, label_precision + label_recall
            )
        results[task] = task_result
    return results


def json_safe(value):
    if isinstance(value, float) and (math.isnan(value) or math.isinf(value)):
        return None
    if isinstance(value, dict):
        return {key: json_safe(item) for key, item in value.items()}
    if isinstance(value, list):
        return [json_safe(item) for item in value]
    return value


RESET_GENERATION_CHECKPOINT = False
# Number of newly generated records per evaluation batch. Metrics are updated after every batch.
GENERATION_EVALUATION_BATCH_SIZE = 8
PREDICTION_CHECKPOINT_PATH = OUTPUT_ROOT / "test_predictions.jsonl"
PARTIAL_METRICS_PATH = OUTPUT_ROOT / "evaluation_metrics_partial.json"
FINAL_METRICS_PATH = OUTPUT_ROOT / "evaluation_metrics.json"
METRICS_HISTORY_PATH = OUTPUT_ROOT / "evaluation_metrics_history.jsonl"


def evaluation_key_from_record(record):
    # Ground truth is included so that changed test annotations are not mistaken for completed records.
    return json.dumps(
        [str(record["id"]), record["task"], str(record["image_path"]), record["answer"]],
        ensure_ascii=False,
        separators=(",", ":"),
    )


def evaluation_key_from_row(row):
    if row.get("evaluation_key"):
        return row["evaluation_key"]
    return json.dumps(
        [str(row["id"]), row["task"], str(row["image_path"]), row["ground_truth"]],
        ensure_ascii=False,
        separators=(",", ":"),
    )


def load_prediction_checkpoint(path):
    rows_by_key = {}
    invalid_lines = 0
    if not path.exists():
        return rows_by_key
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            try:
                row = json.loads(line)
                key = evaluation_key_from_row(row)
                if not isinstance(row.get("generated_text"), str):
                    raise ValueError("missing generated_text")
                rows_by_key[key] = row
            except Exception as exc:
                invalid_lines += 1
                print(f"Ignoring invalid checkpoint line {line_number}: {exc}")
    print(f"Loaded {len(rows_by_key)} completed predictions from {path}")
    if invalid_lines:
        print(f"Ignored {invalid_lines} invalid or partially written checkpoint lines")
    return rows_by_key


def write_json_atomic(path, payload):
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    with temporary_path.open("w", encoding="utf-8") as handle:
        json.dump(json_safe(payload), handle, indent=2)
        handle.flush()
        os.fsync(handle.fileno())
    temporary_path.replace(path)


def write_jsonl_atomic(path, rows):
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    with temporary_path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")
        handle.flush()
        os.fsync(handle.fileno())
    temporary_path.replace(path)


def append_jsonl_durable(path, row):
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(json_safe(row), ensure_ascii=False) + "\n")
        handle.flush()
        os.fsync(handle.fileno())


def ordered_completed_rows(records, rows_by_key):
    return [rows_by_key[key] for key in map(evaluation_key_from_record, records) if key in rows_by_key]


def metric_payload(rows, complete, batch_update_index):
    return {
        "evaluation_complete": complete,
        "batch_update_index": batch_update_index,
        "evaluation_batch_size": GENERATION_EVALUATION_BATCH_SIZE,
        "completed_records": len(rows),
        "total_records": len(test_records),
        "remaining_records": len(test_records) - len(rows),
        "completion_fraction": safe_divide(len(rows), len(test_records)),
        "test_teacher_forced_loss": globals().get("test_teacher_forced_loss"),
        "generation_metrics": evaluate_predictions(rows),
        "coordinate_scale": COORDINATE_SCALE,
        "iou_thresholds": IOU_THRESHOLDS,
    }


prediction_rows = []
if RUN_GENERATION_EVALUATION:
    if RESET_GENERATION_CHECKPOINT:
        for path in [PREDICTION_CHECKPOINT_PATH, PARTIAL_METRICS_PATH, FINAL_METRICS_PATH, METRICS_HISTORY_PATH]:
            if path.exists():
                path.unlink()
        print("Removed the previous generated-box evaluation checkpoint.")

    rows_by_key = load_prediction_checkpoint(PREDICTION_CHECKPOINT_PATH)
    expected_keys = [evaluation_key_from_record(record) for record in test_records]
    completed_count = sum(key in rows_by_key for key in expected_keys)
    if GENERATION_EVALUATION_BATCH_SIZE < 1:
        raise ValueError("GENERATION_EVALUATION_BATCH_SIZE must be at least 1")
    metrics_update_index = 0
    if METRICS_HISTORY_PATH.exists():
        with METRICS_HISTORY_PATH.open("r", encoding="utf-8") as handle:
            metrics_update_index = sum(bool(line.strip()) for line in handle)
    # Continue an interrupted partial batch instead of starting its batch count over.
    new_records_in_batch = completed_count % GENERATION_EVALUATION_BATCH_SIZE
    # Repair a truncated final line and discard stale/duplicate records before appending.
    write_jsonl_atomic(PREDICTION_CHECKPOINT_PATH, ordered_completed_rows(test_records, rows_by_key))
    print(f"Resuming generated-box evaluation at {completed_count}/{len(test_records)} completed records")

    trainer.model.eval()
    with PREDICTION_CHECKPOINT_PATH.open("a", encoding="utf-8") as checkpoint_handle:
        for record, key in zip(test_records, expected_keys):
            if key in rows_by_key:
                continue

            generated = generate_record(trainer.model, record)
            parsed, parse_error = None, None
            try:
                parsed = extract_json_object(generated)
            except Exception as exc:
                parse_error = str(exc)

            row = {
                "evaluation_key": key,
                "id": record["id"],
                "study_id": record["study_id"],
                "subject_id": record["subject_id"],
                "image_path": record["image_path"],
                "task": record["task"],
                "ground_truth": record["answer"],
                "generated_text": generated,
                "parsed_prediction": parsed,
                "parse_error": parse_error,
            }

            checkpoint_handle.write(json.dumps(row, ensure_ascii=False) + "\n")
            checkpoint_handle.flush()
            rows_by_key[key] = row
            completed_count += 1
            new_records_in_batch += 1

            if new_records_in_batch >= GENERATION_EVALUATION_BATCH_SIZE or completed_count == len(test_records):
                os.fsync(checkpoint_handle.fileno())
                prediction_rows = ordered_completed_rows(test_records, rows_by_key)
                batch_is_final = completed_count == len(test_records)
                metrics_update_index += 1
                current_metrics = metric_payload(
                    prediction_rows,
                    complete=batch_is_final,
                    batch_update_index=metrics_update_index,
                )
                # Keep both a latest-state file and an append-only history after every batch.
                write_json_atomic(PARTIAL_METRICS_PATH, current_metrics)
                write_json_atomic(FINAL_METRICS_PATH, current_metrics)
                append_jsonl_durable(METRICS_HISTORY_PATH, current_metrics)
                new_records_in_batch = 0
                print(
                    f"Evaluation batch {metrics_update_index}: "
                    f"{completed_count}/{len(test_records)} records; metrics saved to {FINAL_METRICS_PATH}"
                )

    prediction_rows = ordered_completed_rows(test_records, rows_by_key)
    evaluation_complete = len(prediction_rows) == len(test_records)
    if not evaluation_complete:
        raise RuntimeError(f"Evaluation checkpoint contains only {len(prediction_rows)}/{len(test_records)} records")

    # Compact the checkpoint atomically into current test-set order.
    write_jsonl_atomic(PREDICTION_CHECKPOINT_PATH, prediction_rows)

    final_metrics = metric_payload(
        prediction_rows,
        complete=True,
        batch_update_index=metrics_update_index,
    )
    write_json_atomic(FINAL_METRICS_PATH, final_metrics)
    if PARTIAL_METRICS_PATH.exists():
        PARTIAL_METRICS_PATH.unlink()
    print(json.dumps(json_safe(final_metrics["generation_metrics"]), indent=2))

## 11. Inference with the saved adapter

For deployment, load the same MedGemma base model, attach the saved PEFT adapter, and use conversation(record, include_assistant=False) with either task. Predicted normalized boxes can be converted back to source pixels with denormalize_bbox.

Recommended reporting:

- anatomy and pathology precision, recall, and F1 at IoU 0.25 and 0.50
- mean matched IoU
- per-class metrics for sufficiently frequent anatomy/pathology labels
- valid-JSON rate
- pathology image-label F1 and normality accuracy
- results by projection, source dataset, localization quality, and pathology frequency

Methodological cautions:

- Split by patient or subject, never by individual image.
- Keep normal images as genuine negative pathology examples.
- Do not treat mRALE 0 as a universally normal CXR.
- Avoid crop/flip augmentation unless boxes and laterality are transformed together.
- Large anatomy boxes can dominate token frequency; report pathology localization separately.
- Generated boxes are research outputs and require external validation before clinical use.